In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#TODO: SACAR LOS VALORES DE DIFERENTE TASA DE CRECIMIENTO

In [2]:
excel = pd.ExcelFile('Data/data_important.xlsx')
excel.sheet_names

['Proteins',
 'ProteinRead',
 'Metabolites',
 'MetaboliteRead',
 'MetQ',
 'Fluxes',
 'FluxesRead',
 'FluxQ',
 'Extracelular',
 'ExtracelularRead']

In [3]:
proteins = excel.parse('ProteinRead', index_col=0)
metabolites = excel.parse('MetaboliteRead', index_col=0)
fluxes = excel.parse('FluxesRead', index_col=0)
external_metabolites = excel.parse('ExtracelularRead', index_col=0)

In [ ]:
# important enzymes
important_enzymes = [
    'pgi',
    'pfkB',
    'fbaA',
    'tpiA',
    'gapA',
    'pgk',
    'gpmA',
    'eno',
]

prot_filter = proteins.index.isin(important_enzymes)
prot_fil = proteins[prot_filter]

balanced_metabolites = [
    'Glucose 6-phosphate', # si g6p
    'Fructose 6-phosphate', # si f6p
    'Fructose 1,6-diphosphate', # si fbp
    'Dihydroxyacetone phosphate', # si dhap
    'Glyceraldehyde 3-phosphate', # no g3p
    '1,3-Bisphosphoglycerate', # no pgp
    '3-Phosphoglycerate', # si 3pg
    '2-Phosphoglycerate', # no 2pg
    'Phosphoenolpyruvate', # si pep
]

imbalanced_metabolites = [
    'ATP',
    'ADP',
    'NAD',
    'NADH',
    'Inorganic phosphate',
    'Pyruvate',
]

balanced_metabolites_filter = metabolites.index.isin(balanced_metabolites)
imbalanced_metabolites_filter = metabolites.index.isin(imbalanced_metabolites)

balanced_mets_df = metabolites[balanced_metabolites_filter]
imbalanced_mets_df = metabolites[imbalanced_metabolites_filter]


external_glucose = external_metabolites.index.isin(['Glucose'])
external_glucose_df = external_metabolites[external_glucose]
imbalanced_mets_df = pd.concat([imbalanced_mets_df, external_glucose_df])

important_fluxes = [
    'Glucose + PEP -> G6P + PYR', # pts
    'G6P <-> F6P', # pgi
    'F6P -> F1,6P', # pfkA
    'F1,6P -> DHAP + G3P', # fbaA
    'DHAP -> G3P', # tpiA
    'G3P -> PGP', # gapA
    'PGP -> 3PG', # pgk
    '3PG <-> 2PG', # gpmA
    '2PG <-> PEP', # eno

]
flux_filter = fluxes.index.isin(important_fluxes)

def batch_replace(x):
    if type(x) == str:
        return x.replace(' ', '')

    return x

flux_fil = fluxes[flux_filter].apply(lambda x: x.fillna(0))
flux_fil = flux_fil.apply(lambda x: x.replace('-', 0))

# important external metabolite
important_external_met= [
    'Glucose',
]
external_met_filter = external_metabolites.index.isin(important_external_met)
external_met_fil = external_metabolites[external_met_filter]
###

# Drop conditions where any enzyme measurement is missing
prot_fil = prot_fil.loc[:, prot_fil.notna().all(axis=0)]

columns_consensus = balanced_mets_df.columns.intersection(imbalanced_mets_df.columns).intersection(flux_fil.columns).intersection(prot_fil.columns)
balanced_mets_df = balanced_mets_df[columns_consensus]
imbalanced_mets_df = imbalanced_mets_df[columns_consensus]
flux_fil = flux_fil[columns_consensus]
prot_fil = prot_fil[columns_consensus]

In [ ]:
balanced_mets_df
dict_balanced = {
    'Phosphoenolpyruvate': 'C_pep',
    'Dihydroxyacetone phosphate': 'C_dhap',
    '3-Phosphoglycerate': 'C_3pg',
    'Fructose 6-phosphate': 'C_f6p',
    'Fructose 1,6-diphosphate': 'C_fbp',
    'Glucose 6-phosphate': 'C_g6p',
}
balanced_mets_df['VarNames'] = balanced_mets_df.index.map(dict_balanced)
balanced_mets_df.index = balanced_mets_df['VarNames']
dict_imbalanced = {
    'ATP': 'C_atp',
    'ADP': 'C_adp',
    'Glucose': 'C_glc',
    'NAD': 'C_nad',
    'NADH': 'C_nadh',
    'Inorganic phosphate': 'C_pi',
    'Pyruvate': 'C_pyr',
}
imbalanced_mets_df['VarNames'] = imbalanced_mets_df.index.map(dict_imbalanced)
imbalanced_mets_df.index = imbalanced_mets_df['VarNames']

In [6]:
dict_proteins = {
    'pgi': 'Pgi',
    'pfkB': 'PfkB',
    'fbaA': 'FbaA',
    'tpiA': 'TpiA',
    'gapA': 'GapA',
    'pgk': 'Pgk',
    'gpmA': 'GpmA',
    'eno': 'Eno',
}
prot_fil['VarNames'] = prot_fil.index.map(dict_proteins)
prot_fil
prot_fil.index = prot_fil['VarNames']

In [7]:
dict_flux_name = {
    'Glucose + PEP -> G6P + PYR': 'v_pts',
    'G6P <-> F6P': 'v_pgi',
    'F6P -> F1,6P': 'v_pfkB',
    'F1,6P -> DHAP + G3P': 'v_fbaA',
    'DHAP -> G3P': 'v_tpiA',
    'G3P -> PGP': 'v_gapA',
    'PGP -> 3PG': 'v_pgk',
    '3PG <-> 2PG': 'v_gpmA',
    '2PG <-> PEP': 'v_eno',
}
flux_fil['VarNames'] = flux_fil.index.map(dict_flux_name)
flux_fil
flux_fil.index = flux_fil['VarNames']

In [8]:
flux_fil.index

Index(['v_pts', 'v_pgi', 'v_pfkB', 'v_fbaA', 'v_tpiA', 'v_gapA', 'v_pgk',
       'v_gpmA', 'v_eno'],
      dtype='str', name='VarNames')

In [9]:
# to csv
balanced_mets_df.drop(columns=['VarNames'], inplace=True)
imbalanced_mets_df.drop(columns=['VarNames'], inplace=True)
prot_fil.drop(columns=['VarNames'], inplace=True)
flux_fil.drop(columns=['VarNames'], inplace=True)
balanced_mets_df.to_csv('Data/balanced_metabolites.csv')
imbalanced_mets_df.to_csv('Data/imbalanced_metabolites.csv')
prot_fil.to_csv('Data/important_proteins.csv')
flux_fil.to_csv('Data/important_fluxes.csv')

In [10]:
N = np.array(
        [
            # pts  pgi  pfk  fba  tpi  gap  pgk  gpm  eno
            [0 , 0 , 0 , 0 , 0 , 0 , 0 , 1 , -1],  # 2pg
            [0 , 0 , 0 , 0 , 0 , 0 , 1 , -1, 0 ],  # 3pg
            [0 , 0 , 0 , 1 , -1, 0 , 0 , 0 , 0 ],  # dhap
            [0 , 1 , -1, 0 , 0 , 0 , 0 , 0 , 0 ],  # f6p
            [0 , 0 , 1 , -1, 0 , 0 , 0 , 0 , 0 ],  # fbp
            [0 , 0 , 0 , 1 , 1 , -1, 0 , 0 , 0 ],  # g3p
            [1 , -1, 0 , 0 , 0 , 0 , 0 , 0 , 0 ],  # g6p
            [-1, 0 , 0 , 0 , 0 , 0 , 0 , 0 , 1 ],  # pep
            [0 , 0 , 0 , 0 , 0 , 1 , -1, 0 , 0 ],  # pgp
        ]
    )

metabolites = ["2pg", "3pg", "dhap", "f6p", "fbp", "g3p", "g6p", "pep", "pgp"]
reactions = ["pts", "pgi", "pfk", "fba", "tpi", "gap", "pgk", "gpm", "eno"]

S = pd.DataFrame(N, index=metabolites, columns=reactions)

In [11]:
df_cellular_needs = {}
for conditions in flux_fil.columns:
    flux = flux_fil[conditions].values
    needs = S @ flux
    df_cellular_needs[conditions] = needs
df_cellular_needs = pd.DataFrame(df_cellular_needs, index=metabolites)
df_cellular_needs.index = df_cellular_needs.index.map({
    '2pg': 'C_2pg',
    '3pg': 'C_3pg',
    'dhap': 'C_dhap',
    'f6p': 'C_f6p',
    'fbp': 'C_fbp',
    'g3p': 'C_g3p',
    'g6p': 'C_g6p',
    'pep': 'C_pep',
    'pgp': 'C_pgp',
})
df_cellular_needs.to_csv('Data/cellular_needs.csv')



In [12]:
# measurement errors
df_metQ = excel.parse('MetQ', index_col=0)
balanced_mets_Q = df_metQ[df_metQ.index.isin(balanced_metabolites)]
df_FluxQ = excel.parse('FluxQ', index_col=0)
flux_Q = df_FluxQ[df_FluxQ.index.isin(important_fluxes)]

# transform indices mets
balanced_mets_Q.index = balanced_mets_Q.index.map(dict_balanced)
balanced_mets_Q.drop(columns=['Ave', 'CV'], inplace=True)
balanced_mets_Q.to_csv('Data/balanced_metabolites_Q.csv')

# transform indices fluxes  
flux_Q.index = flux_Q.index.map(dict_flux_name)
flux_Q = flux_Q['STD']
flux_Q.to_csv('Data/important_fluxes_Q.csv')

In [13]:
flux_Q

Sample ID
v_pts     0.312143
v_pgi     0.626764
v_pfkB    0.406177
v_fbaA    0.406177
v_tpiA    0.406177
v_gapA    0.735681
v_pgk     0.735681
v_gpmA    0.736497
v_eno     0.736497
Name: STD, dtype: float64